# MSR – RQ2: does writing a rule down change agent behavior?

Idea: inside the **same repository**, compare agent PRs opened while the rule was **not** in the context file with agent PRs opened while it **was**.
If violations drop once the rule is written down, that is evidence agents follow it.

1. Reuse the RQ1 pipeline (`fetch_data.ipynb`)
2. Build one row per (PR, rule) and mark whether the rule was active when the PR was opened
3. Keep only repos that have agent PRs on both sides
4. Label each row with the RQ1 checkers
5. Violation rate per repo, without vs with the rule
6. Main result: how many repos improved + Wilcoxon signed-rank test
7. Check with a regression (conditional logistic, grouped by repo)
8. Human control for commit format: did humans change in the same way?

In [1]:
import io
import json
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.discrete.conditional_models import ConditionalLogit

## 1. Reuse the RQ1 pipeline

Runs every code cell of `fetch_data.ipynb` that comes **before** its section "## 8. RQ1".
We stop there so this notebook never overwrites your hand labels in `package_manager_violations.csv`.
Their printed output is hidden to keep this notebook short.

After this cell we have `prs` (one row per agent PR with `rules`, `changed_files`, `added_files`, `commit_messages`, `context_hashes`) and all the checker functions.

In [2]:
with open("fetch_data.ipynb") as notebook_file:
    fetch_data = json.load(notebook_file)

for cell in fetch_data["cells"]:
    source = "".join(cell["source"])
    if source.startswith("## 8."):
        break
    if cell["cell_type"] == "code":
        with redirect_stdout(io.StringIO()):
            exec(source)

print(len(prs), "agent PRs with a context file in", prs.repo.nunique(), "repos")

23422 agent PRs with a context file in 1326 repos


## 2. One row per (PR, rule)

Every agent PR gets one row for each rule we study.
`rule_active` = was the rule in the context file when the PR was opened?

`changelog` is left out: too few repos have PRs on both sides.

In [3]:
RQ2_RULES = ["commit_format", "package_manager", "required_tests"]

tables = []
for rule in RQ2_RULES:
    table = prs.copy()
    table["rule"] = rule
    table["rule_active"] = [rule in rules for rules in table.rules]
    tables.append(table)

pairs = pd.concat(tables, ignore_index=True)
pairs = pairs[pairs.changed_files.notna()]

print(len(pairs), "(PR, rule) rows")

68496 (PR, rule) rows


## 3. Keep repos with agent PRs on both sides

A repo is used for a rule only if it has at least `MIN_PRS_PER_SIDE` agent PRs **without** the rule and at least as many **with** it.
Usually "without" means "before the rule was added", but it also includes the rare case where a rule was later removed.

In [4]:
MIN_PRS_PER_SIDE = 5

sides = pairs.groupby(["rule", "repo"]).rule_active.agg(with_rule="sum", total="count")
sides["without_rule"] = sides.total - sides.with_rule

enough_on_both_sides = (sides.without_rule >= MIN_PRS_PER_SIDE) & (sides.with_rule >= MIN_PRS_PER_SIDE)
sides = sides[enough_on_both_sides]

usable_repos = sides.reset_index()[["rule", "repo"]]
pairs = pairs.merge(usable_repos, on=["rule", "repo"])

sides.groupby("rule").agg(
    repos=("total", "count"),
    prs_without_rule=("without_rule", "sum"),
    prs_with_rule=("with_rule", "sum"),
)

,repos,prs_without_rule,prs_with_rule
rule,,,
commit_format,16,392,652
package_manager,20,542,632
required_tests,18,461,744


## 4. Label every row

We use the RQ1 checkers. One change is needed for **package manager**:
the RQ1 checker reads the required manager (e.g. `pnpm`) from the context file, but PRs without the rule have no such text.
So for each repo we take the manager named the **first time** the rule appeared, and check all PRs of that repo against it.

In [5]:
def first_required_managers(repo):
    repo_prs = prs[prs.repo == repo]
    has_rule = ["package_manager" in rules for rules in repo_prs.rules]
    first_pr_with_rule = repo_prs[has_rule].sort_values("created_at").iloc[0]
    return required_managers(context_text(first_pr_with_rule.context_hashes))


package_manager_repos = pairs.loc[pairs.rule == "package_manager", "repo"].unique()
REPO_MANAGERS = {repo: first_required_managers(repo) for repo in package_manager_repos}
REPO_MANAGERS

{'azure/azure-sdk-for-js': {'pnpm'},
 'remotion-dev/remotion': {'pnpm'},
 'microsoft/typespec': {'pnpm'},
 'oxc-project/oxc': {'pnpm'},
 'vltpkg/vltpkg': {'pnpm'},
 'significant-gravitas/autogpt': {'pnpm'},
 'polarsource/polar': {'pnpm', 'uv'},
 'storybookjs/storybook': {'yarn'},
 'open-metadata/openmetadata': {'yarn'},
 'onlook-dev/onlook': {'bun'},
 'eggjs/egg': {'pnpm'},
 'langfuse/langfuse': {'pnpm'},
 'different-ai/note-companion': {'pnpm'},
 'elizaos/eliza': {'bun'},
 'trycompai/comp': {'bun'},
 'stack-auth/stack-auth': {'pnpm'},
 'langfuse/langfuse-docs': {'pnpm'},
 'ceedaragents/cyrus': {'pnpm'},
 'getsentry/sentry-mcp': {'pnpm'},
 'thirdweb-dev/js': {'pnpm'}}

In [6]:
def check_package_manager_for_repo(pr):
    required = REPO_MANAGERS[pr.repo]
    ecosystems = {ECOSYSTEM[manager] for manager in required}

    changed = lockfile_managers(pr.changed_files, ecosystems)
    added = lockfile_managers(pr.added_files, ecosystems)

    if added - required:
        return "violated"
    if changed and changed <= required:
        return "complied"
    return "not_applicable"


RQ2_CHECKERS = {
    "commit_format": check_commit_format,
    "package_manager": check_package_manager_for_repo,
    "required_tests": check_required_tests,
}

pairs["label"] = [RQ2_CHECKERS[row.rule](row) for row in pairs.itertuples()]

pd.crosstab([pairs.rule, pairs.rule_active], pairs.label)

label                        complied  not_applicable  violated
rule            rule_active                                    
commit_format   False             278               0       114
                True              594               0        58
package_manager False              84             455         3
                True               94             533         5
required_tests  False             155             154       152
                True              445             110       189

## 5. Violation rate per repo, without vs with the rule

`not_applicable` rows are dropped (same as RQ1).
A repo needs at least `MIN_CHECKED_PER_SIDE` labeled PRs on each side, otherwise one PR could swing its rate from 0% to 100%.

`change` < 0 means fewer violations once the rule was written down.

In [7]:
MIN_CHECKED_PER_SIDE = 3

checked = pairs[pairs.label != "not_applicable"].copy()
checked["violated"] = checked.label == "violated"
checked["side"] = ["with_rule" if active else "without_rule" for active in checked.rule_active]

rates = checked.pivot_table(index=["rule", "repo"], columns="side", values="violated", aggfunc="mean")
counts = checked.pivot_table(index=["rule", "repo"], columns="side", values="violated", aggfunc="count")

enough_labeled = (counts.without_rule >= MIN_CHECKED_PER_SIDE) & (counts.with_rule >= MIN_CHECKED_PER_SIDE)

repo_rates = rates[enough_labeled].copy()
repo_rates["change"] = repo_rates.with_rule - repo_rates.without_rule
repo_rates["prs_without_rule"] = counts.without_rule
repo_rates["prs_with_rule"] = counts.with_rule

repo_rates.round(2)

side                                                     with_rule  \
rule            repo                                                 
commit_format   514-labs/moosestack                           0.52   
                coder/coder                                   0.50   
                comfy-org/comfyui_frontend                    0.50   
                daveskender/stock.indicators                  0.01   
                different-ai/note-companion                   0.64   
                finaegis/core-banking-prototype-laravel       0.00   
                metamask/metamask-mobile                      0.00   
                newfuture/ddns                                0.10   
                promptfoo/promptfoo                           0.03   
                rnwood/smtp4dev                               0.00   
                shelf-nu/shelf.nu                             0.00   
                significant-gravitas/autogpt                  0.14   
                switchbacktech/compass                        0.27   
                theepicsaxguy/homelab                         0.00   
                theopenco/llmgateway                          0.03   
                usesend/usesend                               0.05   
package_manager azure/azure-sdk-for-js                        0.11   
                different-ai/note-companion                   0.00   
                eggjs/egg                                     0.00   
                elizaos/eliza                                 0.00   
                onlook-dev/onlook                             0.00   
                remotion-dev/remotion                         0.00   
                significant-gravitas/autogpt                  0.00   
required_tests  567-labs/instructor                           0.42   
                azuread/microsoft-identity-web                0.12   
                calcitem/sanmill                              1.00   
                chrxh/alien                                   0.20   
                colinhacks/zod                                0.17   
                dannymac180/meta-agent                        0.33   
                daveskender/stock.indicators                  0.03   
                dotcms/core                                   0.00   
                elizaos/eliza                                 0.00   
                getsentry/sentry                              0.45   
                jdereg/java-util                              0.55   
                langfuse/langfuse                             0.62   
                newfuture/ddns                                0.04   
                prefecthq/prefect                             0.00   
                promptfoo/promptfoo                           0.21   
                rustfs/rustfs                                 0.50   

side                                                     without_rule  change  \
rule            repo                                                            
commit_format   514-labs/moosestack                              0.72   -0.20   
                coder/coder                                      0.29    0.21   
                comfy-org/comfyui_frontend                       0.55   -0.05   
                daveskender/stock.indicators                     0.21   -0.20   
                different-ai/note-companion                      0.31    0.33   
                finaegis/core-banking-prototype-laravel          0.00    0.00   
                metamask/metamask-mobile                         0.00    0.00   
                newfuture/ddns                                   0.46   -0.36   
                promptfoo/promptfoo                              0.00    0.03   
                rnwood/smtp4dev                                  0.44   -0.44   
                shelf-nu/shelf.nu                                0.50   -0.50   
                significant-gravitas/autogpt                     0.29   -0.1

## 6. Main result

Every repo counts once, so one big repo cannot dominate.

- `median_without_rule` / `median_with_rule`: typical repo violation rate on each side
- `repos_improved` / `repos_worse` / `repos_unchanged`: direction of the change
- `wilcoxon_p`: Wilcoxon signed-rank test on the paired repo rates. Small (< 0.05) = the change is unlikely to be chance.
  Repos with no change are ignored by the test. With few repos the test has little power, so also look at the counts.

In [8]:
def summarize(rule_rates):
    improved = (rule_rates.change < 0).sum()
    worse = (rule_rates.change > 0).sum()
    unchanged = len(rule_rates) - improved - worse

    if improved + worse == 0:
        p_value = np.nan
    else:
        p_value = wilcoxon(rule_rates.with_rule, rule_rates.without_rule).pvalue

    return pd.Series({
        "repos": len(rule_rates),
        "median_without_rule": rule_rates.without_rule.median(),
        "median_with_rule": rule_rates.with_rule.median(),
        "repos_improved": improved,
        "repos_worse": worse,
        "repos_unchanged": unchanged,
        "wilcoxon_p": p_value,
    })


main_result = repo_rates.groupby(level="rule").apply(summarize)
main_result.round(3)

,repos,median_without_rule,median_with_rule,repos_improved,repos_worse,repos_unchanged,wilcoxon_p
rule,,,,,,,
commit_format,16.0,0.286,0.037,10.0,4.0,2.0,0.056
package_manager,7.0,0.000,0.000,3.0,1.0,3.0,0.625
required_tests,16.0,0.435,0.206,9.0,6.0,1.0,0.047


## 7. Regression check

Conditional logistic regression grouped by repo: it only compares PRs **within the same repo**, like the test above,
but uses every PR instead of one rate per repo.

- `odds_ratio` < 1: the rule lowers the chance of a violation (e.g. 0.4 = odds of violating are 60% lower)
- `odds_ratio` > 1: more violations after the rule
- `ci_low` / `ci_high`: 95% confidence interval. If it contains 1, the effect is not clear.

Repos where every PR has the same outcome add no information and are skipped by the model automatically.

In [9]:
def rule_effect(rows):
    model = ConditionalLogit(
        endog=rows.violated.astype(int),
        exog=rows[["rule_active"]].astype(int),
        groups=rows.repo,
    ).fit(disp=False)

    ci_low, ci_high = np.exp(model.conf_int().loc["rule_active"])

    return pd.Series({
        "prs": len(rows),
        "repos": rows.repo.nunique(),
        "odds_ratio": np.exp(model.params["rule_active"]),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": model.pvalues["rule_active"],
    })


paired_repos = repo_rates.reset_index()[["rule", "repo"]]
model_rows = checked.merge(paired_repos, on=["rule", "repo"])

regression = pd.DataFrame({rule: rule_effect(rows) for rule, rows in model_rows.groupby("rule")}).T
regression.round(3)

/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 2 groups and 52 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)
/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 3 groups and 63 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)
/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 1 groups and 15 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)


,prs,repos,odds_ratio,ci_low,ci_high,p_value
commit_format,1044.0,16.0,0.291,0.188,0.451,0.000
package_manager,117.0,7.0,0.399,0.035,4.572,0.460
required_tests,891.0,16.0,0.458,0.291,0.720,0.001


## 8. Human control (commit format only)

If humans **also** started following the format after the rule appeared, the agent change may just be the project changing,
not the agent reading the rule.

AIDev v4 has no human PRs, so we use `human_pull_request` from **AIDev v3**. It has titles but no changed files,
so this works only for commit format.

To compare fairly, both groups are judged on the **PR title only** (humans have no commit messages here).
This is stricter than the RQ1 checker, which also accepted a matching commit message.

In [10]:
human_prs = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/human_pull_request.parquet",
    columns=["id", "title", "created_at", "repo_url"],
)
human_prs["repo"] = human_prs.repo_url.str.replace("https://api.github.com/repos/", "").str.lower()
human_prs["created_at"] = pd.to_datetime(human_prs.created_at, utc=True)

commit_format_repos = paired_repos.loc[paired_repos.rule == "commit_format", "repo"]
human_prs = human_prs[human_prs.repo.isin(commit_format_repos)].copy()

human_prs["context_hashes"] = [context_files_at(repo, when) for repo, when in zip(human_prs.repo, human_prs.created_at)]
human_prs = human_prs[human_prs.context_hashes.str.len() > 0].copy()
human_prs["rule_active"] = ["commit_format" in stated_rules(hashes) for hashes in human_prs.context_hashes]
human_prs["author"] = "human"

agent_prs = pairs[(pairs.rule == "commit_format") & pairs.repo.isin(commit_format_repos)].copy()
agent_prs["author"] = "agent"

columns = ["id", "repo", "title", "rule_active", "author"]
title_check = pd.concat([agent_prs[columns], human_prs[columns]], ignore_index=True)
title_check["violated"] = [not follows_format(title) for title in title_check.title]

print(len(human_prs), "human PRs in", human_prs.repo.nunique(), "of the", len(commit_format_repos), "commit format repos")

151 human PRs in 5 of the 16 commit format repos


Violation rate (title only) per author, without vs with the rule, and how many PRs each cell is based on:

In [11]:
pd.concat({
    "violation_rate": title_check.pivot_table(index="author", columns="rule_active", values="violated", aggfunc="mean"),
    "prs": title_check.pivot_table(index="author", columns="rule_active", values="violated", aggfunc="count"),
}, axis=1).round(2)

violation_rate         prs      
rule_active          False True  False True 
author                                      
agent                 0.42  0.15   392   652
human                 0.03  0.16   107    44

Difference-in-differences model, again grouped by repo:

- `rule_active`: change for **humans** when the rule appears
- `rule_active_and_agent`: how much **more** agents change than humans. This is the key number:
  odds ratio < 1 with a CI below 1 = agents react to the written rule beyond what the project as a whole did.

If there are very few human PRs above, this model is unreliable (wide CIs) and should be reported as a pilot only.

In [12]:
title_check["is_agent"] = (title_check.author == "agent").astype(int)
title_check["rule_active"] = title_check.rule_active.astype(int)
title_check["rule_active_and_agent"] = title_check.rule_active * title_check.is_agent

did_model = ConditionalLogit(
    endog=title_check.violated.astype(int),
    exog=title_check[["rule_active", "is_agent", "rule_active_and_agent"]],
    groups=title_check.repo,
).fit(disp=False)

ci = np.exp(did_model.conf_int())
pd.DataFrame({
    "odds_ratio": np.exp(did_model.params),
    "ci_low": ci[0],
    "ci_high": ci[1],
    "p_value": did_model.pvalues,
}).round(3)

,odds_ratio,ci_low,ci_high,p_value
rule_active,4.018,0.888,18.182,0.071
is_agent,12.440,3.555,43.534,0.000
rule_active_and_agent,0.058,0.012,0.284,0.000
